# Generating games with GAMUT

[GAMUT](http://gamut.stanford.edu/) is a suite of parameterised game generators covering a wide range of game families studied in the game theory literature. Written in Java, GAMUT can generate instances of 35 game classes, including random games, coordination games, covariant games, voting games, and many more.

PyGambit's `generate_gamut` function calls GAMUT as an external subprocess and returns the resulting game as a `Game` object, ready for analysis with PyGambit's full suite of tools. Before running this tutorial, you will need Java and `gamut.jar` installed; see the [catalog documentation](../../catalog.html#catalog-gamut) for full installation instructions.

> **Note:** The cell outputs in this notebook were generated locally. To reproduce them, update the `gamut_jar` argument in each cell to the path of your local `gamut.jar`.

This tutorial covers:
- Generating classic two-player games with no additional parameters
- Generating parameterised random normal-form games
- Exploring how the covariance parameter in `CovariantGame` affects equilibrium structure
- Generating multi-player games
- Controlling payoff normalisation and obtaining integer payoffs


In [1]:
import pygambit as gbt

## Classic two-player games

Many of GAMUT's game classes correspond directly to well-known games from the game theory literature and require no additional parameters to generate. Here we generate Battle of the Sexes, a canonical 2×2 coordination game.

In Battle of the Sexes, two players must independently decide whether to go to the Opera or a Football match. Both prefer to attend the same event, but player 1 prefers the Opera while player 2 prefers Football. The game has two pure-strategy Nash equilibria — both go to the Opera, or both go to Football — and one mixed-strategy equilibrium.

We use the `normalize`, `min_payoff`, and `max_payoff` parameters to rescale payoffs to the range [0, 100], making them easier to read:


In [2]:
g_bos = gbt.catalog.generate_gamut(
    "BattleOfTheSexes",
    params={"normalize": True, "min_payoff": 0, "max_payoff": 100},
    gamut_jar="~/Downloads/gamut.jar",
)
g_bos.title = "Battle of the Sexes"
for i, label in enumerate(["Opera", "Football"]):
    g_bos.players[0].strategies[i].label = label
    g_bos.players[1].strategies[i].label = label
g_bos


Game(title='Battle of the Sexes')

We can inspect the payoff matrices directly using `to_arrays`:


In [3]:
p1, p2 = g_bos.to_arrays(dtype=float)
print("Player 1 payoffs:\n", p1)
print("\nPlayer 2 payoffs:\n", p2)


Player 1 payoffs:
 [[5.9929944932832235 0.0]
 [0.0 100.0]]

Player 2 payoffs:
 [[100.0 0.0]
 [0.0 5.9929944932832235]]


Now let's compute all Nash equilibria using the linear complementarity method, which is well-suited to two-player games:


In [4]:
result_bos = gbt.nash.lcp_solve(g_bos)
result_bos.equilibria


[[[Rational(1, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1)]],
 [[Rational(11985988986566447, 211985988986566447), Rational(200000000000000000, 211985988986566447)], [Rational(200000000000000000, 211985988986566447), Rational(11985988986566447, 211985988986566447)]],
 [[Rational(0, 1), Rational(1, 1)], [Rational(0, 1), Rational(1, 1)]]]

As expected, the solver finds three equilibria: two pure-strategy equilibria (both play Opera, or both play Football) and one mixed-strategy equilibrium in which each player randomises between the two options.


## Parameterised random normal-form games

Most GAMUT game classes accept parameters to control the number of players, the number of actions, and the structure of payoffs. These are passed as a dictionary to the `params` argument; each key maps to a GAMUT command-line flag.

`RandomGame` is the most general generator: payoffs are drawn independently and uniformly at random from a fixed range. It serves as a useful baseline for testing algorithms. The `players` parameter controls the number of players, and `actions` controls the number of actions per player. When `actions` is a list, each element gives the action count for the corresponding player, allowing asymmetric games:


In [5]:
g_rand = gbt.catalog.generate_gamut(
    "RandomGame",
    params={"players": 2, "actions": [3, 4]},
    gamut_jar="~/Downloads/gamut.jar",
)
g_rand.title = "Random Game (2 players, 3x4)"
g_rand


Game(title='Random Game (2 players, 3x4)')

In [6]:
p1, p2 = g_rand.to_arrays(dtype=float)
print("Player 1 payoffs:\n", p1)
print("\nPlayer 2 payoffs:\n", p2)


Player 1 payoffs:
 [[7.09879029828295 27.689537794842863 -22.938467441167546
  42.53072437064441]
 [82.25942400799374 -69.1686432869949 96.1110308181407 -79.64396641034494]
 [-9.314488467453216 -7.739226846459999 -83.75467277721359
  62.6622977326341]]

Player 2 payoffs:
 [[32.09734856539009 34.12451016558521 10.426071633781461 27.7180677867141]
 [63.65136522074536 64.14059528213303 -69.44692451793762
  9.130751590062275]
 [-1.253625072125189 64.00556804214489 13.173753520003785
  -22.167235616074493]]


In [7]:
gbt.nash.lcp_solve(g_rand).equilibria[0]


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(0, 1), Rational(1, 1), Rational(0, 1), Rational(0, 1)]]

## Covariant games

`CovariantGame` generates two-player games in which the degree of alignment between players' interests is controlled by a covariance parameter `r`.

- When `r = 1` the game is a common-payoff game: both players receive identical payoffs.
- When `r = 0` payoffs are independent — equivalent to `RandomGame`.
- As `r` approaches `-1` (the minimum for a two-player game) the game approaches a zero-sum game.

Let's compare an equilibrium under high positive covariance (nearly a coordination game) with one under negative covariance (a more adversarial setting):


In [8]:
g_cov_pos = gbt.catalog.generate_gamut(
    "CovariantGame",
    params={"players": 2, "actions": [3, 3], "r": 0.8},
    gamut_jar="~/Downloads/gamut.jar",
)
g_cov_pos.title = "Covariant Game (r=0.8)"
eqm_pos = gbt.nash.lcp_solve(g_cov_pos).equilibria[0]
eqm_pos


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)]]

In [9]:
g_cov_neg = gbt.catalog.generate_gamut(
    "CovariantGame",
    params={"players": 2, "actions": [3, 3], "r": -0.5},
    gamut_jar="~/Downloads/gamut.jar",
)
g_cov_neg.title = "Covariant Game (r=-0.5)"
eqm_neg = gbt.nash.lcp_solve(g_cov_neg).equilibria[0]
eqm_neg


[[Rational(9681608339803400, 125915468067226217), Rational(0, 1), Rational(116233859727422817, 125915468067226217)], [Rational(0, 1), Rational(170889485790150613, 2476491141850744413), Rational(2305601656060593800, 2476491141850744413)]]

With high positive covariance (`r = 0.8`) the game is close to a coordination game, so an equilibrium in pure or near-pure strategies is typical. With negative covariance (`r = -0.5`) the game is more adversarial, making genuinely mixed equilibria more likely.


## Multi-player games

GAMUT includes several n-player game families. Here we use `MajorityVoting`, a model of a committee in which each player votes for one of several candidates. The candidate who receives the most votes wins, and each player has a privately known preferred candidate.

Voting games naturally lend themselves to pure-strategy analysis — each player simply votes for their preferred candidate. `gbt.nash.enumpure_solve` searches exhaustively for pure-strategy Nash equilibria and works for games with any number of players:


In [10]:
g_mv = gbt.catalog.generate_gamut(
    "MajorityVoting",
    params={"players": 3, "actions": 3},
    gamut_jar="~/Downloads/gamut.jar",
)
g_mv.title = "Majority Voting (3 players, 3 candidates)"
g_mv


Game(title='Majority Voting (3 players, 3 candidates)')

In [13]:
result_mv = gbt.nash.enumpure_solve(g_mv)
result_mv.equilibria[0]


[[Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)], [Rational(1, 1), Rational(0, 1), Rational(0, 1)]]

## Payoff normalisation and integer payoffs

By default, GAMUT draws payoffs uniformly from the range [-100, 100]. Four global parameters let you rescale and discretise payoffs:

| Parameter | Type | Effect |
|---|---|---|
| `normalize` | boolean flag | Enable rescaling to [`min_payoff`, `max_payoff`] |
| `min_payoff` | number | Lower bound after normalisation |
| `max_payoff` | number | Upper bound after normalisation |
| `int_payoffs` | boolean flag | Round all payoffs to integers |

Boolean flags are passed with value `True`; the flag is then included on the GAMUT command line without a value token (e.g. `{"normalize": True}` becomes `-normalize`, `{"int_payoffs": True}` becomes `-int_payoffs`).

Integer payoffs are useful when working with Gambit's exact rational arithmetic:


In [12]:
g_int = gbt.catalog.generate_gamut(
    "RandomGame",
    params={
        "players": 2,
        "actions": 3,
        "normalize": True,
        "min_payoff": 0,
        "max_payoff": 10,
        "int_payoffs": True,
    },
    gamut_jar="~/Downloads/gamut.jar",
)
g_int.title = "Random Game (integer payoffs, 0-10)"
p1, p2 = g_int.to_arrays()
print("Player 1 payoffs:\n", p1)
print("\nPlayer 2 payoffs:\n", p2)


Player 1 payoffs:
 [[Rational(48751, 1) Rational(39877, 1) Rational(15655, 1)]
 [Rational(45311, 1) Rational(82068, 1) Rational(8227, 1)]
 [Rational(56758, 1) Rational(35125, 1) Rational(80353, 1)]]

Player 2 payoffs:
 [[Rational(8674, 1) Rational(0, 1) Rational(100000, 1)]
 [Rational(45750, 1) Rational(15592, 1) Rational(8896, 1)]
 [Rational(24363, 1) Rational(50767, 1) Rational(96708, 1)]]
